[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/AM/blob/main/02_kNN.ipynb)

# Introdução ao Aprendizado de Máquina  
## Aula 2 — k-Nearest Neighbors (k-NN)

**Professor:** Diogo Ferreira de Lima Silva (UFF)  
**Curso:** Engenharia de Produção  

Este notebook foi estruturado para acompanhar a aula sobre métodos baseados em distância. 
 
A ideia aqui não é apenas **usar** o `sklearn`, mas entender **o que o algoritmo está fazendo**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# 1. Um problema simples de classificação

Vamos começar com um exemplo **artificial e pequeno**, apenas para construir intuição.

Cada objeto possui dois atributos:
- $x_1$
- $x_2$

E cada objeto pertence a uma de duas classes:
- classe **0**
- classe **1**

A pergunta central é:

> Dado um novo ponto $\mathbf{x}$, como decidir seu rótulo?


In [ ]:
# Abaixo veremos uma base de treinamento muito pequena, apenas para fins didáticos
X_train_small = np.array([
    [1.0, 1.0],
    [1.5, 1.8],
    [2.0, 1.0],
    [4.0, 4.0],
    [4.5, 5.0],
    [5.0, 4.2]
])
 

print(f"Matriz de Atributos do Conjunto de Treinamento:{X_train_small}")


In [ ]:
y_train_small = np.array([0, 0, 0, 1, 1, 1])
print(f"Vetor de Rótulos do Conjunto de Treinamento:{y_train_small}")

In [ ]:
# Novo ponto a ser classificado
x_test_small = np.array([2.7, 2.4])
print(f"Exemplo de teste: {x_test_small}")

### Visualizando o nosso conjunto de dados

In [ ]:
plt.figure(figsize=(6, 5))

# Desenhando os exemplos de treinamento
for classe in np.unique(y_train_small):
    pontos = X_train_small[y_train_small == classe]
    plt.scatter(pontos[:, 0], pontos[:, 1], s=90, label=f'Classe {classe}')

# Desenhando o exemplo de teste
plt.scatter(x_test_small[0], x_test_small[1], s=180, marker='*', label='Novo ponto')


plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Exemplo inicial para intuição do k-NN')
plt.legend()
plt.grid(True)
plt.show()


- Os pontos já rotulados formam nosso **conjunto de treinamento**.  

- A estrela representa um **novo exemplo** cujo rótulo queremos prever.

- O k-NN parte da seguinte ideia: **exemplos parecidos tendem a ter rótulos parecidos**.

- Assim, precisamos de alguma forma medir a distância entre os pontos.


# 2. Distância entre exemplos

Nesta aula, usaremos inicialmente a **distância euclidiana** entre dois vetores
$\mathbf{x}^{(i)}$ e $\mathbf{x}^{(j)}$:

$$
d\big(\mathbf{x}^{(i)}, \mathbf{x}^{(j)}\big)
=
\sqrt{\sum_{k=1}^{n}
\left(x_k^{(i)} - x_k^{(j)}\right)^2}
$$

Em Python, podemos implementá-la diretamente.


In [ ]:
def distancia_euclidiana(x1, x2):
    x1 = np.array(x1, dtype=float)
    x2 = np.array(x2, dtype=float)
    return np.sqrt(np.sum((x1 - x2)**2))


In [ ]:
# Vamos calcular manualmente a distância do novo ponto para cada ponto do conjunto de treinamento.

distancias = [distancia_euclidiana(x_test_small, x) for x in X_train_small]

distancias

Para tabelar os valores obtidos e ter uma melhor visualização, podemos usar a biblioteca pandas:

In [ ]:
tabela_distancias = pd.DataFrame({
    'indice_treino': np.arange(len(X_train_small)),
    'x1_treino': X_train_small[:, 0],
    'x2_treino': X_train_small[:, 1],
    'classe': y_train_small,
    'distancia_ate_x_teste': distancias
}).sort_values('distancia_ate_x_teste')

tabela_distancias


### Interpretação

A tabela acima mostra exatamente o que o algoritmo faz no caso mais simples:
1. calcula a distância do novo ponto a **todos** os pontos de treinamento;
2. ordena os exemplos do mais próximo para o mais distante;
3. decide o rótulo com base nos vizinhos mais próximos.

No caso do **1-NN**, usamos apenas o mais próximo.  
No caso do **k-NN**, usamos os $k$ mais próximos.


## Exercício 1 — distância

Sem executar a próxima célula, tente responder:

1. Qual é o vizinho mais próximo de `x_test_small`?
2. Qual seria a classe prevista pelo **1-NN**?
3. Se usarmos **k = 3**, qual será a classe prevista?

Depois, confira com o código.


In [ ]:
# Conferência do Exercício 1
vizinho_mais_proximo = tabela_distancias.iloc[0]

#1-NN
print('Vizinho mais próximo:')
print(vizinho_mais_proximo["indice_treino"].astype(int))

print("----------------------------------------")

k3 = tabela_distancias.head(3)
#3-NN
print('Vizinhos mais próximos:')
print(list(k3["indice_treino"].astype(int)))
print('\nClasses dos 3 vizinhos mais próximos:', list(k3['classe']))
print('Classe majoritária entre os 3 mais próximos:', k3['classe'].mode()[0])


# 3. Implementação manual do 1-NN

Agora vamos implementar o algoritmo 1-NN do zero.

Lembre que, nesse caso:

- o “treinamento” consiste basicamente em **armazenar os dados**;
- a predição exige comparar o novo exemplo com todos os exemplos de treino.


In [ ]:
def predizer_1nn(X_train, y_train, x_test):
    distancias = [distancia_euclidiana(x_test, x) for x in X_train]
    indice_mais_proximo = np.argmin(distancias)
    return y_train[indice_mais_proximo], indice_mais_proximo, distancias


In [ ]:
classe_prevista, indice_vizinho, distancias = predizer_1nn(X_train_small, y_train_small, x_test_small)

print(f'Índice do vizinho mais próximo: {indice_vizinho}')
print(f'Classe prevista pelo 1-NN: {classe_prevista}')


## Comentário importante

Perceba que o 1-NN é um método extremamente simples:
- ele não ajusta uma equação;
- ele não estima coeficientes;
- ele não aprende uma árvore;
- ele apenas compara o novo ponto com os dados já observados.

Por isso o k-NN é frequentemente chamado de **método preguiçoso** (*lazy learning*).


## Exercício 2 — implementação manual

Complete a função abaixo para implementar novamente o 1-NN. Use termos diferentes dos utilizados acima.

Dica: você precisará:
- calcular todas as distâncias;
- encontrar o índice da menor distância;
- retornar o rótulo correspondente.


In [ ]:
def meu_1nn(X_train, y_train, x_test):
    # TODO:
    # 1. calcule as distâncias do ponto x_test até todos os pontos de X_train
    # 2. encontre o índice da menor distância
    # 3. retorne o rótulo correspondente em y_train
    pass

# Teste sua implementação aqui:
# meu_1nn(X_train_small, y_train_small, x_test_small)


# 4. Implementação manual do k-NN

No k-NN, em vez de olhar apenas para o vizinho mais próximo, consideramos os $k$ vizinhos mais próximos.

Em classificação, a regra mais comum é a **votação majoritária**:

$$
\hat{y}(\mathbf{x})
=
\arg\max_{c \in \mathcal{Y}}
\sum_{i \in N_k(\mathbf{x})} \mathbf{1}(y^{(i)} = c)
$$

onde $N_k(\mathbf{x})$ é o conjunto dos índices dos $k$ vizinhos mais próximos de $\mathbf{x}$.


In [ ]:
def predizer_knn(X_train, y_train, x_test, k=3):
    distancias = np.array([distancia_euclidiana(x_test, x) for x in X_train])
    indices_ordenados = np.argsort(distancias)
    indices_vizinhos = indices_ordenados[:k]
    classes_vizinhas = y_train[indices_vizinhos]

    contagem = Counter(classes_vizinhas)
    classe_prevista = contagem.most_common(1)[0][0]

    return classe_prevista, indices_vizinhos, distancias


In [ ]:
# Para k=2, temos:

y_hat, vizinhos, distancias = predizer_knn (X_train_small, y_train_small, x_test_small, k=2)

print(y_hat)

In [ ]:
for k in [1, 3, 5]:
    classe_prevista, indices_vizinhos, distancias = predizer_knn(
        X_train_small, y_train_small, x_test_small, k=k
    )
    print(f'k = {k}')
    print('  índices vizinhos =', indices_vizinhos)
    print('  classes vizinhas =', y_train_small[indices_vizinhos])
    print('  classe prevista  =', classe_prevista)
    print()


### Discussão

Ao variar $k$, o modelo pode mudar bastante:

- **$k$ pequeno**: decisão muito local, sensível a ruído;
- **$k$ grande**: decisão mais estável, mas pode ignorar estruturas locais.

Esse é um dos pontos centrais do método.


## Exercício 3 — votação

Usando o exemplo pequeno acima, responda:

1. Para `k = 4`, quais são as classes dos cinco vizinhos mais próximos?
2. Qual é a classe majoritária?
3. O que pode acontecer se escolhermos um valor de $k$ muito grande?

Escreva sua resposta em texto na célula abaixo.


In [ ]:
# Escreva o seu código para obter as informações acima


# 5. Visualização das fronteiras de decisão

Uma grande vantagem didática do k-NN é que conseguimos visualizar suas regiões de decisão em duas dimensões.

Vamos construir uma função auxiliar para desenhar a fronteira de decisão.


In [ ]:
def plotar_fronteira_knn(X_train, y_train, k=1, titulo=None):
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )

    pontos_grade = np.c_[xx.ravel(), yy.ravel()]
    previsoes = []
    for ponto in pontos_grade:
        classe_prevista, _, _ = predizer_knn(X_train, y_train, ponto, k=k)
        previsoes.append(classe_prevista)

    Z = np.array(previsoes).reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.25)

    for classe in np.unique(y_train):
        pontos = X_train[y_train == classe]
        plt.scatter(pontos[:, 0], pontos[:, 1], s=90, label=f'Classe {classe}')

    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(titulo if titulo is not None else f'Fronteira de decisão do k-NN (k={k})')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plotar_fronteira_knn(X_train_small, y_train_small, k=3, titulo='Fronteira de decisão — k-NN com k=3')


# 6. Aplicação ao conjunto de dados Iris

Agora vamos trabalhar com um conjunto de dados real muito usado em introduções a aprendizado de máquina: o **Iris dataset**.

Cada exemplo corresponde a uma flor, com quatro atributos numéricos:
- comprimento da sépala
- largura da sépala
- comprimento da pétala
- largura da pétala

E o rótulo é a espécie:
- setosa
- versicolor
- virginica


In [ ]:
from sklearn.datasets import load_iris

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print('Shape de X:', X.shape)
print('Shape de y:', y.shape)
print('Nomes das classes:', iris.target_names)
print('Nomes dos atributos:', iris.feature_names)


In [ ]:
y

In [ ]:
df_iris = pd.DataFrame(X, columns=iris.feature_names)
df_iris['classe'] = y
df_iris.head(10)


## Separação entre treino e teste

Como discutido em aula, não devemos avaliar o modelo nos mesmos dados usados para treiná-lo.

Vamos separar os dados em:
- **treinamento**: usados para ajustar o modelo;
- **teste**: usados para avaliar a capacidade de generalização.


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)


## Normalização

Como o k-NN depende de distâncias, diferenças de escala entre atributos podem afetar fortemente o resultado.

Vamos aplicar a normalização min-max com base **apenas no conjunto de treinamento**.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_test_norm = scaler.transform(X_test)

X_train_norm[:5]

### Comentário conceitual

A normalização deve ser ajustada usando apenas os dados de treinamento.  
Depois, a mesma transformação é aplicada aos dados de teste.

Isso evita vazamento de informação do conjunto de teste para o treinamento.


# 7. Aplicando o k-NN com `scikit-learn`

Agora que já entendemos o algoritmo conceitualmente, vamos usar a implementação pronta do `sklearn`.

A sequência é:

1. criar o modelo;
2. ajustá-lo com os dados de treinamento;
3. fazer previsões para novos dados.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

# No k-NN, "fit" significa basicamente armazenar os exemplos de treinamento
knn.fit(X_train_norm, y_train)

# Previsão para os exemplos de teste
y_pred = knn.predict(X_test_norm)

print('Acurácia no teste:', accuracy_score(y_test, y_pred))


In [ ]:
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred))


In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))


### 📊 Avaliação de desempenho do modelo

O relatório acima apresenta diferentes métricas utilizadas para avaliar o desempenho de um classificador. Cada linha corresponde a uma classe (setosa, versicolor, virginica), e as métricas são calculadas comparando os rótulos previstos com os rótulos reais.

#### 🔹 Precision (Precisão)
A precisão mede, dentre todas as vezes que o modelo **previu uma determinada classe**, quantas estavam corretas.

$\text{precision} = \frac{\text{verdadeiros positivos}}{\text{verdadeiros positivos} + \text{falsos positivos}}$

👉 Interpretação:
- Alta precisão → quando o modelo diz "é dessa classe", geralmente está certo.
- Baixa precisão → o modelo "confunde" essa classe com outras.

---

#### 🔹 Recall (Revocação ou Sensibilidade)
O recall mede, dentre todos os exemplos que **realmente pertencem a uma classe**, quantos foram corretamente identificados.

$\text{recall} = \frac{\text{verdadeiros positivos}}{\text{verdadeiros positivos} + \text{falsos negativos}}$

👉 Interpretação:
- Alto recall → o modelo consegue encontrar a maioria dos exemplos da classe.
- Baixo recall → o modelo "deixa passar" muitos exemplos dessa classe.

---

#### 🔹 F1-score
O F1-score é a média harmônica entre precision e recall.

$F1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$

👉 Interpretação:
- Combina precisão e recall em uma única métrica.
- Útil quando queremos equilibrar os dois.

---

#### 🔹 Support
Número de exemplos reais de cada classe no conjunto de teste.

👉 Importante:
- Ajuda a entender se os dados estão balanceados.
- Métricas em classes com pouco suporte podem ser menos confiáveis.

---

### 🔹 Métricas globais

#### ✔️ Accuracy (Acurácia)
Proporção total de acertos do modelo:

$\text{accuracy} = \frac{\text{número de acertos}}{\text{total de exemplos}}$

👉 No exemplo: 0.97 → o modelo acertou 97% dos casos.


⚠️ Limitação:
- Pode ser enganosa em datasets desbalanceados.

---

#### ✔️ Macro average
Média simples das métricas entre as classes (todas têm o mesmo peso).

👉 Útil quando queremos tratar todas as classes igualmente.

---

#### ✔️ Weighted average
Média ponderada pelo número de exemplos de cada classe (support).

👉 Útil quando as classes têm tamanhos diferentes.

---

### 📌 Interpretação do resultado

- O modelo teve desempenho **excelente** em todas as classes.
- Pequenas diferenças aparecem, por exemplo:
  - A classe *virginica* teve recall um pouco menor (0.90), indicando que alguns exemplos dessa classe foram confundidos com outras.
- Como os dados estão balanceados (support = 10 para cada classe), macro avg e weighted avg são iguais.

---


# 8. O efeito da normalização

No Iris, os atributos já estão em escalas relativamente parecidas.  
Por isso, o efeito da normalização pode não parecer tão dramático.

Para entender melhor **por que a escala importa**, vamos criar um exemplo artificial no qual um atributo domina numericamente o outro.


In [ ]:
X_escala = np.array([
    [1.0, 1000.0],
    [2.0, 1020.0],
    [1.2, 980.0],
    [8.0, 1005.0],
    [9.0, 995.0],
    [8.5, 1010.0]
])

y_escala = np.array([0, 0, 0, 1, 1, 1])

x_novo = np.array([3.0, 1006.0])

print('Distâncias sem normalizar:')
for i, x in enumerate(X_escala):
    print(i, distancia_euclidiana(x_novo, x))


In [ ]:
scaler_demo = MinMaxScaler()
X_escala_norm = scaler_demo.fit_transform(X_escala)
x_novo_norm = scaler_demo.transform([x_novo])[0]

print('Distâncias após normalização:')
for i, x in enumerate(X_escala_norm):
    print(i, distancia_euclidiana(x_novo_norm, x))


### Interpretação

Sem normalização, o atributo de maior escala pode dominar a distância.  
Com normalização, os atributos passam a contribuir de forma mais comparável.

Essa é uma das razões pelas quais a etapa de pré-processamento é tão importante em métodos baseados em distância.


# 9. Escolhendo o valor de $k$

Uma pergunta importante é:

> qual valor de $k$ devemos usar?

Não existe uma resposta universal.  
Em geral, escolhemos $k$ com base em avaliação empírica.

Vamos testar vários valores de $k$ no Iris.


In [ ]:
resultados = []

for k in range(1, 21):
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_train_norm, y_train)
    y_pred_k = modelo.predict(X_test_norm)
    acc = accuracy_score(y_test, y_pred_k)
    resultados.append((k, acc))

df_resultados = pd.DataFrame(resultados, columns=['k', 'acuracia'])
df_resultados


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(df_resultados['k'], df_resultados['acuracia'], marker='o')
plt.xlabel('k')
plt.ylabel('Acurácia no conjunto de teste')
plt.title('Desempenho do k-NN para diferentes valores de k')
plt.grid(True)
plt.show()


In [ ]:
melhor_ = df_resultados.loc[df_resultados['acuracia'].idxmax()]
print(f"Melhor resultado observado: k = {int(melhor_['k'])}, acurácia = {melhor_['acuracia']:.4f}")


### Discussão

Ao variar $k$, alteramos o compromisso entre:
- sensibilidade local;
- robustez a ruído;
- complexidade da fronteira de decisão.

Na prática, é comum usar **validação cruzada** para escolher esse hiperparâmetro.


## Exercício — investigação prática

Teste os valores `k = 1`, `k = 5` e `k = 15` e responda:

1. Qual deles apresentou melhor desempenho?
2. O valor de $k$ que teve melhor desempenho aqui necessariamente será o melhor em qualquer base?
3. Como essa discussão se relaciona com o *No Free Lunch Theorem*?

Você pode usar a célula abaixo para experimentar.


In [ ]:
# Espaço livre para o Exercício 7

# Sugestão:
# for k in [1, 5, 15]:
#     ...


# 10. Comparando previsão para um único exemplo

Também podemos pedir ao modelo que classifique um único exemplo do conjunto de teste.


In [ ]:
indice = 0
x_exemplo = X_test_norm[indice].reshape(1, -1)
classe_real = y_test[indice]
classe_prevista = knn.predict(x_exemplo)[0]

print('Classe real    :', iris.target_names[classe_real])
print('Classe prevista:', iris.target_names[classe_prevista])
print('Atributos normalizados do exemplo:', X_test_norm[indice])


## Exercício  — exploração livre

Escolha alguns outros índices do conjunto de teste e verifique:

- quais exemplos foram classificados corretamente;
- quais exemplos foram classificados incorretamente;
- se há algum padrão nos erros.


In [ ]:
# Espaço livre para o Exercício 8


# 11. Síntese

Até aqui, vimos que o k-NN:

- armazena exemplos rotulados;
- mede distâncias entre o novo exemplo e os exemplos de treinamento;
- escolhe os vizinhos mais próximos;
- usa votação para decidir a classe em problemas de classificação.

Do ponto de vista conceitual, o k-NN pode ser visto como uma forma de estimar localmente:

$$
P(Y = c \mid X = \mathbf{x})
$$

pela frequência relativa das classes na vizinhança de $\mathbf{x}$.


# 12. Próxima aula

Na próxima aula, estudaremos o **Naive Bayes**.

A ideia será novamente classificar um exemplo usando probabilidades, mas agora por um caminho diferente:

- no k-NN, usamos uma noção de **proximidade local**;
- no Naive Bayes, usaremos um **modelo probabilístico** para estimar probabilidades de classe.

Em outras palavras, continuaremos tentando responder à pergunta:

> dado um vetor de atributos $\mathbf{x}$, qual é a classe mais provável?

Mas mudaremos completamente a forma de fazer essa estimativa.
